# KV-cache eviction and a planted complaint: Colab T4 runbook

**Runtime → Change runtime type → T4 GPU**, then run top to bottom.

Before you start, upload the repo to `MyDrive/kv-complaints`, **with** `results/manifest.json`
and `data/review_texts.json` from your laptop build and **without** the `yelp/` folder.
Results are written to Drive, so a disconnect loses nothing.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/kv-complaints
!ls results/manifest.json data/review_texts.json

Mounted at /content/drive
/content/drive/MyDrive/kv-complaints
data/review_texts.json	results/manifest.json


In [3]:
!pip install -q "kvpress==0.5.5" "transformers==4.57.*" accelerate pytest
!pip freeze > requirements.lock
!nvidia-smi --query-gpu=name,memory.total --format=csv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
name, memory.total [MiB]
Tesla T4, 15360 MiB


If an import fails after pip changed numpy or pandas: **Runtime → Restart session**, re-run the first cell, continue here.

## Git
`run.py` records the commit of every run, and Test 4 checks every committed file. Edit your name and email once.

In [ ]:
!git config --global --add safe.directory '*'
!git config --global user.name "Your Name"
!git config --global user.email "you@example.com"
!test -d .git || git init -q
!git add -A && git commit -qm "Freeze code, lexicon and templates (code drafted with Claude)" || true
!git log --oneline | head -3

## Smoke test: KnormPress at 50% on a ~2k-token dummy context

In [ ]:
import torch, kvpress
from kvpress import KnormPress
from run import load_pipe

pipe = load_pipe()
dummy = ' '.join(['The soup was warm and the staff were kind.'] * 200)
print(pipe(dummy, question='\n\nWhat did people say about the soup?',
           press=KnormPress(compression_ratio=0.5), max_new_tokens=40)['answer'])
print('model revision:', pipe.model.config._commit_hash)
del pipe; torch.cuda.empty_cache()

## Tests 1 to 4
`-rs` lists skipped tests: all four must say passed, none skipped.

In [ ]:
!python -m pytest -q -rs tests

## Calibration
Full cache, 12 planted contexts, Q1 only. If Full mentions the complaint in under 30% or over 95%,
change `N_RECOMMENDATIONS` in `config.py` **once** (to 8 or 3), log it in `logs/decisions.md`, commit, then launch.

In [ ]:
!python run.py --calibrate

## The 240 runs
Roughly 1 to 1.5 h on a T4. If Colab disconnects, re-run the first cell and this one: it resumes.

In [ ]:
!python run.py

In [ ]:
!wc -l data/runs_raw.jsonl

## Redact, label, analyse
1. `redact.py` writes the committable `results/runs.jsonl`.
2. `labels.py` writes `labels/to_label.csv`. Label every row (1 per aspect the advice addresses, else 0)
   and save it as `labels/labels.csv`. Don't open `labels/key.csv`.
3. Run the analysis, pick the primary detector by your pre-registered rule, set `PRIMARY_DETECTOR`
   in `config.py`, and run the analysis again.

In [ ]:
!python redact.py
!python labels.py

In [ ]:
!python analysis.py

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/fig1_mention_rate.png'))
display(Image('results/figures/fig2_retention.png'))

## Final commit
First clear this notebook's outputs (**Edit → Clear all outputs**, then **File → Save**), so no answer text is saved in it.
Files are staged before the tests, so Test 4 checks everything that is about to be committed.

In [ ]:
!git add -A
!python -m pytest -q -rs tests && git commit -qm "Runs, labels, tables and figures" && git tag v-assessment
!git log --oneline | head -3